<a href="https://colab.research.google.com/github/aishwaryasuriyakumar/aishwaryasrcs09-codeboosters-internship-2026/blob/main/DAY_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5: Machine Learning for Data Engineers
## Codeboosters Tech — Data Engineering + GenAI Internship
### Phase 1 Finale

---

**Continuity:** `student_performance.csv` from Day 1 returns as ML training data.

**Complete Pipeline Today:**
```
student_performance.csv
       ↓
[Day 1] Load with Pandas
       ↓
[Day 3] ETL / Clean (nulls, types)
       ↓
[Day 5] Feature Engineering (encode, scale)
       ↓
[Day 5] Train ML Model (Scikit-learn)
       ↓
[Day 5] Evaluate (MAE, RMSE, R²)
       ↓
[Day 5] Predict new student scores
```

---

## Setup — Import All Libraries

In [ ]:
# ============================================================
# CELL 1 — Import all required libraries
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn: Python's standard ML library (pre-installed in Colab)
from sklearn.model_selection import train_test_split
# train_test_split: divides data into training and testing portions

from sklearn.preprocessing import LabelEncoder, StandardScaler
# LabelEncoder: converts text categories to integers (Male→0, Female→1)
# StandardScaler: scales numbers so all features have same range

from sklearn.linear_model import LinearRegression
# LinearRegression: fits a straight-line relationship between features and target

from sklearn.tree import DecisionTreeRegressor
# DecisionTreeRegressor: builds a tree of if/else rules to predict

from sklearn.ensemble import RandomForestRegressor
# RandomForestRegressor: builds 100 decision trees and averages predictions

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# mean_absolute_error (MAE):  average absolute prediction error
# mean_squared_error  (MSE):  average squared error (penalises large mistakes)
# r2_score            (R²):   0.0 to 1.0 quality score (1.0 = perfect)

print('All libraries imported successfully!')

---
## STEP 1 — Load and Inspect the Dataset

Upload `student_performance.csv` to Colab Files panel. This is the same file from Day 1.

In [ ]:
# ============================================================
# CELL 2 — Load the student dataset (Day 1 continuity)
# ============================================================

df = pd.read_csv('student_performance.csv')
# Same CSV from Day 1 — we have come full circle
# Day 1: explored it | Day 2: SQL queries | Day 3: ETL | Day 5: ML

print(f'Dataset loaded: {df.shape[0]} students, {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')
print(f'\nMissing values: {df.isnull().sum().sum()}')
print('\nFirst 5 rows:')
df.head()

In [ ]:
# ============================================================
# CELL 3 — Understand the prediction task
# ============================================================

print('=== ML TASK DEFINITION ===')
print('TARGET   : programming_score (the number we want to predict)')
print('FEATURES : math_score, science_score, english_score,')
print('           attendance_percentage, gender, department')
print()
print('Question: Given a student\'s exam scores and attendance,')
print('          can we predict their programming score?')
print()
print('--- Target column statistics ---')
print(df['programming_score'].describe())
# This tells us the range of values we need to predict
# Mean, min, max help us understand what a 'good' prediction looks like

---
## STEP 2 — Feature Engineering

**Why can't we feed raw data into ML models?**

ML models are mathematical functions. They only understand **numbers**.
Our raw data has:
- Text categories: `gender` = 'Male'/'Female', `department` = 'Computer Science' etc.
- These must be converted to integers before training.

We also need to handle:
- Different scales across features
- Dropping irrelevant identifier columns

In [ ]:
# ============================================================
# CELL 4 — Feature Engineering: Encode categorical columns
# ============================================================

df_ml = df.copy()
# Create a copy for ML processing — never modify the original
# ETL best practice from Day 3: df = raw.copy()

# Encode 'gender' column: text → integer
le_gender = LabelEncoder()
df_ml['gender_encoded'] = le_gender.fit_transform(df_ml['gender'])
# LabelEncoder().fit_transform() does two things:
#   fit:      learns the unique values in the column ('Female','Male')
#   transform: converts them to integers (Female→0, Male→1)
# The mapping is alphabetical: Female=0, Male=1

print(f'Gender encoding: {dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_)))}')
# Shows: {'Female': 0, 'Male': 1}

# Encode 'department' column: text → integer
le_dept = LabelEncoder()
df_ml['department_encoded'] = le_dept.fit_transform(df_ml['department'])

print(f'Department encoding: {dict(zip(le_dept.classes_, le_dept.transform(le_dept.classes_)))}')
# Shows: {'Civil': 0, 'Computer Science': 1, 'Electronics': 2, 'Mechanical': 3}

print('\nNew columns added: gender_encoded, department_encoded')
df_ml[['gender','gender_encoded','department','department_encoded']].head(5)

In [ ]:
# ============================================================
# CELL 5 — Select features and define target
# ============================================================

# FEATURES (X): the input columns the model will learn from
feature_cols = [
    'math_score',
    'science_score',
    'english_score',
    'attendance_percentage',
    'gender_encoded',
    'department_encoded'
]
# We include:
#   - Numeric scores: already numbers, no encoding needed
#   - attendance_percentage: numeric, no encoding needed
#   - gender_encoded, department_encoded: text columns we just encoded
# We EXCLUDE:
#   - student_id: just an identifier, not a meaningful feature
#   - name: text, not informative for prediction
#   - city, admission_year: not related to academic performance
#   - semester: all students are in semester 2 (constant — adds no information)

X = df_ml[feature_cols]
# X is the FEATURE MATRIX: shape (30, 6) — 30 students, 6 features each
# Convention: X is uppercase because it is a matrix

y = df_ml['programming_score']
# y is the TARGET VECTOR: shape (30,) — one score per student
# Convention: y is lowercase because it is a vector

print(f'Feature matrix X shape: {X.shape}  (students × features)')
print(f'Target vector y shape : {y.shape}  (one score per student)')
print(f'\nFeature columns: {feature_cols}')
print(f'Target column  : programming_score')
print(f'\nTarget range: {y.min()} to {y.max()} (mean: {y.mean():.1f})')

In [ ]:
# ============================================================
# CELL 6 — Train/Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
# train_test_split() shuffles the data randomly then divides it:
#   test_size=0.2   = reserve 20% of data for testing (6 students)
#                     the remaining 80% is used for training (24 students)
#   random_state=42 = fixed random seed — ensures the same split every run
#                     Without this, each run gives different train/test students

# WHY split? Because we need unseen data to measure real accuracy.
# If we trained AND tested on the same data, the model would score
# near-perfect by memorizing answers — but fail on new students.
# Analogy: you cannot grade a student using the exact questions they studied.

print(f'Total students    : {len(X)}')
print(f'Training students : {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Testing students  : {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nTraining target range: {y_train.min()} – {y_train.max()}')
print(f'Testing  target range: {y_test.min()}  – {y_test.max()}')

---
## STEP 3 — Scale Features

**Why scale?**

`math_score` ranges 0–100. `gender_encoded` is 0 or 1.
Without scaling, Linear Regression treats `math_score` as 100× more important than `gender_encoded` just because of its size — not because it actually is.
StandardScaler puts all features on equal footing.

In [ ]:
# ============================================================
# CELL 7 — Scale features with StandardScaler
# ============================================================

scaler = StandardScaler()
# StandardScaler transforms each feature to have mean=0 and std=1
# Formula: scaled_value = (original_value - mean) / standard_deviation
# Example: math_score of 85 with mean=76 and std=10 → (85-76)/10 = 0.9

X_train_scaled = scaler.fit_transform(X_train)
# fit_transform on TRAINING data:
#   fit:       calculates mean and std FROM TRAINING DATA ONLY
#   transform: applies the scaling
# CRITICAL: we fit ONLY on training data — the model must not 'see' test data

X_test_scaled = scaler.transform(X_test)
# transform ONLY on test data (not fit_transform!)
# We use the SAME mean and std learned from training data
# This simulates real deployment: new data is scaled using training statistics

print('Features scaled successfully!')
print(f'Training feature mean (should be ≈0): {X_train_scaled.mean():.4f}')
print(f'Training feature std  (should be ≈1): {X_train_scaled.std():.4f}')
print('\nNote: Scaler was fit on training data only — prevents data leakage')

---
## STEP 4 — Train and Evaluate Three Models

We train three models and compare their performance.
The same 80/20 split ensures a fair comparison.

In [ ]:
# ============================================================
# CELL 8 — Train Model 1: Linear Regression
# ============================================================

lr_model = LinearRegression()
# LinearRegression() creates an UNTRAINED model
# At this point the model is like a blank student who has read no examples

lr_model.fit(X_train_scaled, y_train)
# .fit() is the TRAINING step
# The model processes all 24 training students
# It finds the best weights (coefficients) w1, w2, ... w6
# that minimise prediction error
# Formula learned: programming_score ≈ w1*math + w2*science + ... + bias

lr_pred = lr_model.predict(X_test_scaled)
# .predict() applies learned weights to the 6 TEST students
# Returns predicted programming scores
# The model has NEVER seen these 6 students during training

# Evaluate
lr_mae  = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2   = r2_score(y_test, lr_pred)

print('=== Model 1: Linear Regression ===')
print(f'MAE  : {lr_mae:.2f}  (predictions are off by {lr_mae:.1f} points on average)')
print(f'RMSE : {lr_rmse:.2f}  (error with penalty for large mistakes)')
print(f'R²   : {lr_r2:.4f}  ({lr_r2*100:.1f}% of programming score variation explained)')
print()
print('Learned coefficients (importance of each feature):')
for feat, coef in zip(feature_cols, lr_model.coef_):
    print(f'  {feat:<28}: {coef:+.3f}')
# Positive coefficient = higher value → higher predicted score
# Negative coefficient = higher value → lower predicted score
# Larger absolute value = more influence on prediction
print(f'  {"bias (intercept)":<28}: {lr_model.intercept_:+.3f}')

In [ ]:
# ============================================================
# CELL 9 — Train Model 2: Decision Tree
# ============================================================

dt_model = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)
# DecisionTreeRegressor builds a tree of if/else rules:
#   IF math_score > 80 AND attendance > 88 THEN score ≈ 91
#   IF math_score <= 80 THEN score ≈ 65
# max_depth=5: limit tree to 5 levels
#   Without this limit, the tree grows until it memorizes every training
#   student perfectly (overfitting) but fails on new data
# random_state=42: reproducible results

dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)

dt_mae  = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
dt_r2   = r2_score(y_test, dt_pred)

print('=== Model 2: Decision Tree (max_depth=5) ===')
print(f'MAE  : {dt_mae:.2f}')
print(f'RMSE : {dt_rmse:.2f}')
print(f'R²   : {dt_r2:.4f}')

In [ ]:
# ============================================================
# CELL 10 — Train Model 3: Random Forest
# ============================================================

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
# RandomForestRegressor builds MANY decision trees and averages predictions
# n_estimators=100: build 100 different Decision Trees
#   Each tree is trained on a random SUBSET of training students
#   Each tree uses a random SUBSET of features at each split
# Final prediction = average of all 100 tree predictions
# Why better than one tree?
#   Each tree makes different errors. Averaging cancels out random errors.
#   This is called ensemble learning — wisdom of the crowd applied to ML

rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2   = r2_score(y_test, rf_pred)

print('=== Model 3: Random Forest (100 trees) ===')
print(f'MAE  : {rf_mae:.2f}')
print(f'RMSE : {rf_rmse:.2f}')
print(f'R²   : {rf_r2:.4f}')

In [ ]:
# ============================================================
# CELL 11 — Compare all three models side by side
# ============================================================

print('=' * 60)
print('  MODEL COMPARISON')
print('=' * 60)
print(f'{"Model":<20} {"MAE":>8} {"RMSE":>8} {"R²":>8}')
print('-' * 48)

results = [
    ('Linear Regression', lr_mae, lr_rmse, lr_r2),
    ('Decision Tree',     dt_mae, dt_rmse, dt_r2),
    ('Random Forest',     rf_mae, rf_rmse, rf_r2),
]

best_model = None
best_r2 = -999

for name, mae, rmse, r2 in results:
    star = ' ← BEST' if r2 == max(lr_r2, dt_r2, rf_r2) else ''
    print(f'{name:<20} {mae:>8.2f} {rmse:>8.2f} {r2:>8.4f}{star}')
    if r2 > best_r2:
        best_r2 = r2
        best_model = name

print('=' * 60)
print(f'Best model: {best_model} (R² = {best_r2:.4f})')
print()
print('Interpretation:')
print(f'  MAE = {min(lr_mae,dt_mae,rf_mae):.1f} means predictions are off by ~{min(lr_mae,dt_mae,rf_mae):.0f} marks on average')
print(f'  R² = {best_r2:.2f} means {best_r2*100:.0f}% of score variation is explained by our features')

---
## STEP 5 — Visualize Results

In [ ]:
# ============================================================
# CELL 12 — Visualization: 3-panel ML results dashboard
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle(
    'ML Results — Student Programming Score Prediction\n'
    'Trained on student_performance.csv (Day 1 dataset)',
    fontsize=16, fontweight='bold', color='#1E2761', y=1.02
)

# ── Panel 1: Actual vs Predicted (Random Forest) ──
ax1 = axes[0]
ax1.scatter(y_test, rf_pred, color='#27AE60', s=120, zorder=5,
            edgecolors='white', linewidth=1.5, label='Predictions')
# scatter plot: x = actual scores, y = predicted scores
# Perfect model = all points on the diagonal line

min_val = min(y_test.min(), rf_pred.min()) - 3
max_val = max(y_test.max(), rf_pred.max()) + 3
ax1.plot([min_val, max_val], [min_val, max_val],
         'r--', linewidth=2, label='Perfect prediction')
# Diagonal line = where actual == predicted
# Points close to line = good predictions

for i, (actual, pred) in enumerate(zip(y_test, rf_pred)):
    ax1.annotate(f'{actual}→{pred:.0f}',
                 (actual, pred), fontsize=8, ha='left',
                 xytext=(3, 3), textcoords='offset points')

ax1.set_xlabel('Actual Score', fontsize=12)
ax1.set_ylabel('Predicted Score', fontsize=12)
ax1.set_title(f'Actual vs Predicted\n(Random Forest, R²={rf_r2:.3f})',
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3, linestyle='--')

# ── Panel 2: Feature Importance (Random Forest) ──
ax2 = axes[1]
importances = rf_model.feature_importances_
# feature_importances_: how much each feature contributed to predictions
# Values sum to 1.0 (100%)
# Higher = more important for the model's decisions

feat_imp_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=True)

colors = ['#E74C3C' if i > 0.15 else '#4FC3F7'
          for i in feat_imp_df['importance']]
bars = ax2.barh(feat_imp_df['feature'], feat_imp_df['importance'],
                color=colors, edgecolor='white')
for bar in bars:
    ax2.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f'{bar.get_width():.3f}',
             va='center', fontsize=10, fontweight='bold')
ax2.set_title('Feature Importance\n(Random Forest)',
              fontsize=13, fontweight='bold')
ax2.set_xlabel('Importance Score', fontsize=12)
ax2.grid(axis='x', alpha=0.3, linestyle='--')
ax2.set_xlim(0, max(importances) + 0.08)

# ── Panel 3: Model Comparison Bar Chart ──
ax3 = axes[2]
model_names = ['Linear\nRegression', 'Decision\nTree', 'Random\nForest']
r2_scores   = [lr_r2, dt_r2, rf_r2]
bar_colors  = ['#4FC3F7', '#E67E22', '#27AE60']

bars3 = ax3.bar(model_names, r2_scores,
                color=bar_colors, edgecolor='white', linewidth=0.8)
for bar, score in zip(bars3, r2_scores):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{score:.4f}', ha='center', fontsize=11, fontweight='bold')
ax3.set_title('Model Comparison\n(R² Score — higher is better)',
              fontsize=13, fontweight='bold')
ax3.set_ylabel('R² Score', fontsize=12)
ax3.set_ylim(0, 1.15)
ax3.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='0.7 threshold (good)')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('ml_results_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved: ml_results_dashboard.png')

---
## STEP 6 — Predict for New Students

In [ ]:
# ============================================================
# CELL 13 — Predict programming score for new hypothetical students
# ============================================================

# Define new students (not in our dataset)
new_students = pd.DataFrame({
    'math_score':            [88,  62,  75],
    'science_score':         [82,  58,  80],
    'english_score':         [75,  65,  72],
    'attendance_percentage': [92,  70,  85],
    'gender_encoded':        [1,   0,   1 ],   # 1=Male, 0=Female
    'department_encoded':    [1,   2,   0 ],   # 1=CS, 2=Electronics, 0=Civil
})
# These are completely hypothetical students
# The model was never trained on these — truly new predictions

# Scale using the SAME scaler fitted on training data
new_students_scaled = scaler.transform(new_students[feature_cols])
# We must use .transform() (not .fit_transform())
# The scaler already learned the mean and std from training data
# We just apply those same statistics to new data

# Predict with all three models
predictions_lr = lr_model.predict(new_students_scaled)
predictions_dt = dt_model.predict(new_students_scaled)
predictions_rf = rf_model.predict(new_students_scaled)

print('=== PREDICTIONS FOR NEW STUDENTS ===')
print(f'{"":<12} {"Math":>6} {"Sci":>5} {"Eng":>5} {"Att%":>6} | {"Lin.Reg":>8} {"Dec.Tree":>9} {"Rand.Forest":>12}')
print('-' * 72)

student_labels = ['Student A', 'Student B', 'Student C']
for i, label in enumerate(student_labels):
    row = new_students.iloc[i]
    print(f'{label:<12} {row["math_score"]:>6} {row["science_score"]:>5} '
          f'{row["english_score"]:>5} {row["attendance_percentage"]:>6} | '
          f'{predictions_lr[i]:>8.1f} {predictions_dt[i]:>9.1f} {predictions_rf[i]:>12.1f}')

print()
print('Interpretation:')
print('  Student A (high scores + attendance) → predicted to score well')
print('  Student B (lower scores + attendance) → predicted to score lower')
print('  Student C (mixed profile) → mid-range prediction')
print()
print('The model learned these patterns from 24 training students!')

In [ ]:
# ============================================================
# CELL 14 — Predict for a student using YOUR OWN scores
# ============================================================
# Fill in your own scores and see what programming score the model predicts!

YOUR_MATH       = 80   # Replace with your math score (0-100)
YOUR_SCIENCE    = 75   # Replace with your science score (0-100)
YOUR_ENGLISH    = 70   # Replace with your english score (0-100)
YOUR_ATTENDANCE = 88   # Replace with your attendance % (0-100)
YOUR_GENDER     = 1    # 0=Female, 1=Male
YOUR_DEPT       = 1    # 0=Civil, 1=Computer Science, 2=Electronics, 3=Mechanical

my_data = scaler.transform([[
    YOUR_MATH, YOUR_SCIENCE, YOUR_ENGLISH,
    YOUR_ATTENDANCE, YOUR_GENDER, YOUR_DEPT
]])

my_prediction = rf_model.predict(my_data)[0]
# rf_model.predict() returns an array — [0] gets the single value

print(f'Your input: Math={YOUR_MATH}, Science={YOUR_SCIENCE}, '
      f'English={YOUR_ENGLISH}, Attendance={YOUR_ATTENDANCE}%')
print(f'\nPredicted Programming Score: {my_prediction:.1f}')
print(f'(Random Forest model, R²={rf_r2:.3f})')

---
# PHASE 1 FINAL MINI PROJECT

## Choose ONE of the following projects:

### Project Option 1: Student Analytics System
Complete end-to-end pipeline using `student_performance.csv`

### Project Option 2: Weather Intelligence System
Complete end-to-end pipeline using `weather_data.csv` from Day 3

---

## PROJECT 1: Student Analytics System

Build a complete pipeline integrating ALL Phase 1 skills:
`CSV → SQL Analysis → Visualization → ML Prediction → Report`

In [ ]:
# ============================================================
# PROJECT 1 — CELL 15: SQL Analysis Layer
# ============================================================
import sqlite3

# Store student data in SQLite (Day 2 skill)
conn = sqlite3.connect('analytics.db')
df.to_sql('students', conn, if_exists='replace', index=False)

# Run analytical SQL queries
queries = {
    'Top performers': '''
        SELECT name, department,
               math_score + science_score + english_score + programming_score AS total_score
        FROM students
        ORDER BY total_score DESC
        LIMIT 5
    ''',
    'Dept averages': '''
        SELECT department,
               ROUND(AVG(programming_score),2) AS avg_programming,
               ROUND(AVG(attendance_percentage),2) AS avg_attendance,
               COUNT(*) AS student_count
        FROM students
        GROUP BY department
        ORDER BY avg_programming DESC
    ''',
    'At-risk students': '''
        SELECT name, department, programming_score, attendance_percentage
        FROM students
        WHERE attendance_percentage < 80 OR programming_score < 60
        ORDER BY programming_score ASC
    '''
}

sql_results = {}
for title, query in queries.items():
    result = pd.read_sql_query(query, conn)
    sql_results[title] = result
    print(f'\n--- {title} ---')
    print(result.to_string(index=False))

conn.close()

In [ ]:
# ============================================================
# PROJECT 1 — CELL 16: Complete Analytics Dashboard
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(20, 13))
fig.suptitle(
    'Student Analytics System — Phase 1 Final Project\n'
    'Data Engineering + SQL + Visualization + Machine Learning',
    fontsize=18, fontweight='bold', color='#1E2761', y=1.01
)

colors6 = ['#4FC3F7','#00897B','#E67E22','#8E44AD']

# Panel 1: Avg programming score by department
dept_prog = df.groupby('department')['programming_score'].mean().sort_values(ascending=False)
axes[0][0].bar(dept_prog.index, dept_prog.values, color=colors6, edgecolor='white')
for i, (dept, val) in enumerate(dept_prog.items()):
    axes[0][0].text(i, val+0.5, f'{val:.1f}', ha='center', fontweight='bold', fontsize=10)
axes[0][0].set_title('Avg Programming Score\nby Department', fontsize=13, fontweight='bold')
axes[0][0].set_ylabel('Score')
axes[0][0].set_ylim(0,110)
axes[0][0].tick_params(axis='x', labelsize=9)
axes[0][0].grid(axis='y', alpha=0.3, linestyle='--')

# Panel 2: Attendance distribution histogram
axes[0][1].hist(df['attendance_percentage'], bins=8,
                color='#4FC3F7', edgecolor='white', alpha=0.85)
axes[0][1].axvline(df['attendance_percentage'].mean(), color='#E74C3C',
                   linestyle='--', linewidth=2,
                   label=f"Mean: {df['attendance_percentage'].mean():.1f}%")
axes[0][1].set_title('Attendance Distribution', fontsize=13, fontweight='bold')
axes[0][1].set_xlabel('Attendance %')
axes[0][1].set_ylabel('Number of Students')
axes[0][1].legend(fontsize=10)
axes[0][1].grid(axis='y', alpha=0.3, linestyle='--')

# Panel 3: Math vs Programming scatter
colors_dept = {'Computer Science':'#4FC3F7','Electronics':'#00897B',
               'Mechanical':'#E67E22','Civil':'#8E44AD'}
for dept, group in df.groupby('department'):
    axes[0][2].scatter(group['math_score'], group['programming_score'],
                       label=dept, color=colors_dept[dept],
                       s=80, alpha=0.8, edgecolors='white')
axes[0][2].set_title('Math Score vs\nProgramming Score', fontsize=13, fontweight='bold')
axes[0][2].set_xlabel('Math Score')
axes[0][2].set_ylabel('Programming Score')
axes[0][2].legend(fontsize=8, loc='upper left')
axes[0][2].grid(alpha=0.3, linestyle='--')

# Panel 4: Top 10 students horizontal bar
df['total_score'] = df['math_score']+df['science_score']+df['english_score']+df['programming_score']
top10 = df.nlargest(8,'total_score')[['name','total_score','department']].sort_values('total_score')
bar_colors_top = [colors_dept[d] for d in top10['department']]
axes[1][0].barh(top10['name'], top10['total_score'],
                color=bar_colors_top, edgecolor='white')
for bar in axes[1][0].patches:
    axes[1][0].text(bar.get_width()-8, bar.get_y()+bar.get_height()/2,
                    f'{int(bar.get_width())}', va='center', color='white',
                    fontsize=9, fontweight='bold')
axes[1][0].set_title('Top 8 Students\n(Total Score out of 400)', fontsize=13, fontweight='bold')
axes[1][0].set_xlabel('Total Score')
axes[1][0].grid(axis='x', alpha=0.3, linestyle='--')

# Panel 5: Gender distribution pie
gender_counts = df['gender'].value_counts()
axes[1][1].pie(gender_counts.values, labels=gender_counts.index,
               colors=['#4FC3F7','#E67E22'], autopct='%1.0f%%',
               startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1][1].set_title('Gender Distribution', fontsize=13, fontweight='bold')

# Panel 6: ML Predictions vs Actual
axes[1][2].scatter(y_test, rf_pred, color='#27AE60', s=120,
                   edgecolors='white', linewidth=1.5, zorder=5)
mn = min(y_test.min(), rf_pred.min())-3
mx = max(y_test.max(), rf_pred.max())+3
axes[1][2].plot([mn,mx],[mn,mx],'r--',linewidth=2,label='Perfect')
axes[1][2].set_title(f'ML: Actual vs Predicted\nRandom Forest R²={rf_r2:.3f}',
                     fontsize=13, fontweight='bold')
axes[1][2].set_xlabel('Actual Score')
axes[1][2].set_ylabel('Predicted Score')
axes[1][2].legend(fontsize=10)
axes[1][2].grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('student_analytics_system.png', dpi=150, bbox_inches='tight')
plt.show()
print('Student Analytics System dashboard saved!')

In [ ]:
# ============================================================
# PROJECT 1 — CELL 17: Executive Summary Report
# ============================================================

print('=' * 65)
print('   STUDENT ANALYTICS SYSTEM — PHASE 1 FINAL REPORT')
print('   Codeboosters Tech Internship | 5-Day Data Pipeline')
print('=' * 65)

print(f'\n[DATASET]')
print(f'  Students analysed : {len(df)}')
print(f'  Departments       : {df["department"].nunique()}')
print(f'  Data quality      : {df.isnull().sum().sum()} missing values')

print(f'\n[PERFORMANCE INSIGHTS]')
best_dept = df.groupby("department")["programming_score"].mean().idxmax()
print(f'  Best dept (programming): {best_dept}')
top_student = df.loc[df["total_score"].idxmax()]
print(f'  Top student              : {top_student["name"]} — {top_student["total_score"]}/400')
low_att = df[df["attendance_percentage"]<80]
print(f'  Students needing support : {len(low_att)} (attendance < 80%)')

print(f'\n[MACHINE LEARNING RESULTS]')
print(f'  Model             : Random Forest (100 trees)')
print(f'  Target predicted  : programming_score')
print(f'  Training students : {len(X_train)}')
print(f'  Test students     : {len(X_test)}')
print(f'  MAE               : {rf_mae:.2f} marks')
print(f'  R² Score          : {rf_r2:.4f} ({rf_r2*100:.1f}% variance explained)')

feat_imp = dict(zip(feature_cols, rf_model.feature_importances_))
top_feat = max(feat_imp, key=feat_imp.get)
print(f'  Most important feature: {top_feat} ({feat_imp[top_feat]:.3f})')

print(f'\n[PIPELINE SUMMARY]')
for day, task in [
    ('Day 1','Loaded CSV with Pandas, explored 30 students'),
    ('Day 2','Stored in SQLite, wrote SQL queries'),
    ('Day 3','ETL pipeline + Weather API'),
    ('Day 4','PySpark Bronze→Silver→Gold Medallion'),
    ('Day 5','Feature engineering + ML prediction'),
]:
    print(f'  {day}: {task}')

print('\n' + '=' * 65)
print('Phase 1 Complete. Upload to GitHub.')
print('Commit: Add Day 5: ML Pipeline + Phase 1 Final Project')
print('=' * 65)

---
## Practice Questions

**Q1:** What is the difference between supervised and unsupervised ML?

**Q2:** Why can't you feed raw `student_performance.csv` directly into Scikit-learn?

**Q3:** What does `train_test_split()` do and why is it necessary?

**Q4:** A model has R²=0.92 on training data but R²=0.41 on test data. What is this problem?

**Q5:** Write code to train a Random Forest and print its R² score.

**Q6:** Which Day 1–4 step is most critical for ML model quality and why?

---

In [ ]:
# ============================================================
# CELL 18 — Answer Q4 and Q5
# ============================================================

# Answer Q4: Overfitting
print('Answer Q4: OVERFITTING')
print('  R²=0.92 on training means the model memorized training data perfectly.')
print('  R²=0.41 on test means it fails on new, unseen data.')
print('  Fix: Use max_depth on Decision Tree, or use Random Forest (built-in regularization).')
print()

# Answer Q5
print('Answer Q5: Random Forest with R² score')
rf_answer = RandomForestRegressor(n_estimators=100, random_state=42)
rf_answer.fit(X_train_scaled, y_train)
pred_answer = rf_answer.predict(X_test_scaled)
print(f'  R² Score: {r2_score(y_test, pred_answer):.4f}')

---
## Day 5 Summary

| Concept | Key Takeaway |
|---------|-------------|
| Machine Learning | Computers learn patterns from labelled examples |
| Supervised Learning | Input + correct answer → model learns the mapping |
| Regression | Predict a number (score, price, temperature) |
| LabelEncoder | Convert text categories to integers |
| StandardScaler | Scale features to same range (mean=0, std=1) |
| train_test_split | Reserve 20% for testing — prevent memorization |
| model.fit() | Training step — model learns from data |
| model.predict() | Apply learned model to new, unseen data |
| MAE / RMSE | Measure average prediction error |
| R² score | 0–1 quality score (>0.7 = good) |
| Random Forest | 100 trees averaged — best accuracy, resists overfitting |
| feature_importances_ | Which features matter most for predictions |

## GitHub Upload Checklist
- [ ] All cells run without errors
- [ ] Model comparison table printed
- [ ] `ml_results_dashboard.png` saved
- [ ] Phase 1 Final Project complete
- [ ] `student_analytics_system.png` saved
- [ ] Upload: `Add Day 5: ML Pipeline + Phase 1 Final Project`

## Phase 2 Preview (Day 6–10)
Tomorrow we enter **Generative AI** — calling LLM APIs, building chatbots,
semantic search, RAG systems, and AI agents. Your Phase 1 data pipeline skills
feed directly into Phase 2 AI workflows!

---
*Codeboosters Tech | Data Engineering + GenAI Internship | Day 5 of 10 | Phase 1 Complete*